# KHIS Toolkit - Quick Start

Pull, clean, and visualise Kenya health data in minutes

In [ ]:
import os

USE_REAL_KHIS = "hiskenya" in os.getenv("DHIS2_BASE_URL", "").lower()

if USE_REAL_KHIS:
    print("Kenya mode is active: using real KHIS credentials and county-ready data.")
else:
    print("Demo mode is active: using the public DHIS2 demo server. Expect test data and non-Kenya org units.")

In [ ]:
import khis

conn = khis.connect()

In [ ]:
indicators = khis.list_indicators(conn)
indicators.head(10)

In [ ]:
# In real KHIS mode you can request Kenya counties directly.
# In demo mode the public server does not contain Kenya counties, so we use the available demo org units instead.
if USE_REAL_KHIS:
    df_raw = khis.get(
        conn,
        "malaria",
        counties=["Nairobi", "Kisumu", "Mombasa"],
        periods="last_12_months",
    )
else:
    malaria_matches = khis.list_indicators(conn, search="malaria")
    indicator_id = malaria_matches.iloc[0]["id"]
    demo_org_units = conn.get_org_units(level=3)
    if demo_org_units.empty:
        demo_org_units = conn.get_org_units()
    demo_org_units = demo_org_units[["id", "name"]].head(3)
    display(demo_org_units)
    df_raw = conn.get_analytics(
        indicator_ids=indicator_id,
        org_unit_ids=demo_org_units["id"].tolist(),
        periods="LAST_12_MONTHS",
    )

df_raw.head()

In [ ]:
import pandas as pd

df_clean = khis.clean(df_raw)
comparison = pd.concat(
    [
        df_raw.head().reset_index(drop=True).add_prefix("raw_"),
        df_clean.head().reset_index(drop=True).add_prefix("clean_"),
    ],
    axis=1,
)
comparison

In [ ]:
scorecard, summary = khis.quality_report(df_clean)
display(scorecard)
print(summary)

In [ ]:
selected_county = df_clean["org_unit_name"].dropna().iloc[0]
selected_indicator = df_clean["indicator_name"].dropna().iloc[0]
forecast_df = khis.forecast_indicator_series(
    df_clean,
    county=selected_county,
    indicator=selected_indicator,
    weeks_ahead=4,
    method="ensemble",
)
forecast_df.tail(8)

In [ ]:
from dashboard.map import create_county_map

if USE_REAL_KHIS:
    latest_values = (
        df_clean.sort_values(["org_unit_name", "period"])
        .groupby("org_unit_name", as_index=False)
        .tail(1)
        .rename(columns={"org_unit_name": "county", "value": "map_value"})
    )[["county", "map_value"]]
else:
    latest_demo = (
        df_clean.sort_values(["org_unit_name", "period"])
        .groupby("org_unit_name", as_index=False)
        .tail(1)["value"]
        .reset_index(drop=True)
    )
    latest_values = khis.list_counties().head(len(latest_demo))[["name"]].rename(columns={"name": "county"})
    latest_values["map_value"] = latest_demo.values

county_map = create_county_map(latest_values, value_col="map_value", title="KHIS Toolkit county map preview")
county_map

## What Next?

- Get KHIS credentials: khissupport@health.go.ke
- Connect to real Kenya county data
- Read the forecasting notebook: 04_forecasting.ipynb
- Deploy the dashboard: see README deployment section